In [ ]:
# ==================== SETUP & ENVIRONMENT CONFIGURATION ====================
# These imports and configurations are required for all code in this notebook

import os
from dotenv import load_dotenv
import openai

# Load environment variables from .env file
# This file should contain: OPENAI_API_KEY=your_key_here
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

# Set OpenAI API key for authentication
# Without this, all API calls to OpenAI will fail
openai.api_key = os.getenv("OPENAI_API_KEY")

## 🔧 Prerequisites & Installation

### Required Packages

Before running this notebook, install all required packages:

```bash
pip install -r requirements.txt
```

**Or install individually:**
```bash
pip install langchain-core langchain-openai langchain-community openai python-dotenv
```

### Environment Setup

Create a `.env` file in this folder with:
```
OPENAI_API_KEY=your_openai_api_key_here
```

Get your API key from: https://platform.openai.com/api-keys

---

# Week 2: Prompts, Chains & Memory in LangChain

## 📚 Learning Objectives

By the end of this notebook, you will understand:
1. **Prompts**: How to create structured, reusable prompts for LLMs
2. **Chains**: How to connect multiple operations (prompts, LLMs, functions) together
3. **Memory**: How to build chatbots that remember conversation history

---

## 🎯 Why This Matters?

- **Prompts** = How you communicate with LLMs
- **Chains** = How you build complex workflows (not just single-turn interactions)
- **Memory** = How you create conversational AI that feels natural and contextual

Without these three concepts, you can only build basic, stateless question-answer systems.

---

## Overview

This notebook covers three core concepts:
- **Prompts**: Structured ways to format instructions for LLMs
- **Chains**: Connecting multiple operations together (e.g., translate → summarize)
- **Memory**: Keeping track of conversation history for context-aware responses

---

## 📝 Section 1: Basic Prompt Template

A `PromptTemplate` is a structured way to format text inputs for language models. It uses placeholders (variables in curly braces) that get filled in at runtime.

**Why use templates?** 
- Reusable across different inputs
- Easy to modify prompts without changing code
- Professional and maintainable

In [ ]:
# =============== SECTION 1: Basic Prompt Template Demo ===============

# Import PromptTemplate from LangChain
from langchain_core.prompts import PromptTemplate

# Step 1️⃣: Define a template with a placeholder
# The {content} will be replaced when we call format()
template = "Summarize in 1 line: {content}"

# Step 2️⃣: Create a PromptTemplate object
prompt = PromptTemplate.from_template(template)

# Step 3️⃣: Format the template by providing a value for {content}
output = prompt.format(content="AI is transforming every industry.")

print("=" * 50)
print("EXAMPLE: Simple Prompt Template")
print("=" * 50)
print(f"Template: {template}")
print(f"Input: 'AI is transforming every industry.'")
print(f"Formatted Output:\n{output}")
print()


Summarize in 1 line: AI is transforming every industry.


---

## 📝 Section 2: Building Chains - Translate + Summarize

### What is a Chain?

A **chain** is a sequence of operations connected together using the pipe operator `|`. It allows you to:
1. Pass the output of one step as input to the next step
2. Avoid writing complex nested function calls
3. Build reusable workflow components

### The Pipeline Concept

Think of it like an assembly line:
```
Input → Prompt 1 → LLM → Output → Prompt 2 → LLM → Final Output
```

### This Example

We'll build a **Translation + Summarization Pipeline**:
1. Take input text (in any language)
2. Translate it to English
3. Summarize the translated text in one line

**Key Concepts:**
- `|` operator: Chains operations together (from `langchain_core.runnables`)
- Lambda function: Used to transform data between chain steps
- `invoke()`: Method to run the complete pipeline

In [ ]:
# =============== SECTION 2: Chaining - Translate + Summarize ===============

# Import necessary components
from langchain_openai import OpenAI
from langchain_core.prompts import PromptTemplate

print("=" * 60)
print("DEMO: Translation + Summarization Pipeline")
print("=" * 60)
print()

# ============== STEP 1: Initialize the LLM ==============
# OpenAI model for text generation
# "gpt-3.5-turbo-instruct" is an older model good for instructions
llm = OpenAI(model="gpt-3.5-turbo-instruct")
print("✓ LLM initialized: gpt-3.5-turbo-instruct")
print()

# ============== STEP 2: Create the Translation Prompt ==============
translate_prompt = PromptTemplate.from_template(
    "Translate the following text to English:\n\n{text}"
)
print("✓ Translation prompt created")

# ============== STEP 3: Create the Summary Prompt ==============
summary_prompt = PromptTemplate.from_template(
    "Summarize the following English text in 1 line:\n\n{translated_text}"
)
print("✓ Summary prompt created")
print()

# ============== STEP 4: Build individual chains ==============
# Chain = Prompt | LLM
# This means: "Send the formatted prompt to the LLM"
translate_chain = translate_prompt | llm
summary_chain = summary_prompt | llm
print("✓ Individual chains created")

# ============== STEP 5: Combine chains into a pipeline ==============
# Flow: Input → Translate → Extract text → Summarize
# Lambda function: Takes translate_chain output and formats it as input to summary_chain
pipeline = translate_chain | (lambda x: {"translated_text": x}) | summary_chain
print("✓ Pipeline assembled")
print()

# ============== STEP 6: Run the pipeline ==============
input_text = "Bonjour! Je veux apprendre l'IA."
print(f"Input (French): {input_text}")
print()

result = pipeline.invoke({"text": input_text})

print("Output (Translated & Summarized):")
print(result)
print()
print("=" * 60)




"I am interested in learning AI."


---

## 📝 Section 3: Adding Conversation Memory

### Why Memory?

So far, all our chains have been **stateless** (they don't remember previous interactions). A real chatbot needs **state** (memory) to:
- Remember the user's name from earlier
- Reference previous topics in the conversation
- Provide contextual responses

### How Memory Works in LangChain

1. **ChatPromptTemplate**: Structured template for multi-turn conversations
2. **MessagesPlaceholder**: A dynamic slot in the prompt where conversation history goes
3. **ChatMessageHistory**: Storage object that keeps all messages (user & assistant)
4. **RunnableWithMessageHistory**: Wrapper that automatically manages memory

### Memory Architecture

```
User Input
   ↓
Retrieve History from Storage
   ↓
Include History in Prompt: [System Message] + [Old Messages] + [New User Message]
   ↓
Send to LLM
   ↓
LLM generates response (aware of context)
   ↓
Save new message pair to History
   ↓
Return response to user
```

### Key Components Explained

- **`ChatPromptTemplate.from_messages()`**: Creates a multi-turn conversation template
- **`MessagesPlaceholder`**: Variable name for where history will be inserted
- **`ChatOpenAI`**: The LLM that understands conversation context
- **`RunnableWithMessageHistory`**: Connects chain + storage + LLM

In [ ]:
# =============== SECTION 3: Memory Demo - Simple Chatbot ===============

# Import all required components
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

print("=" * 60)
print("DEMO: Chatbot with Conversation Memory")
print("=" * 60)
print()

# ============== STEP 1: Initialize the LLM ==============
# ChatOpenAI is specifically for conversation (vs OpenAI for text completion)
llm = ChatOpenAI(model="gpt-4o-mini")
print("✓ LLM initialized: gpt-4o-mini (chat model)")

# ============== STEP 2: Create a conversation prompt template ==============
# This template includes:
# - MessagesPlaceholder: Will be filled with conversation history
# - User input: The current user message
prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("history"),      # Where past messages will go
    ("user", "{input}")                  # The current user message
])
print("✓ Chat prompt template created (with history placeholder)")

# ============== STEP 3: Create in-memory history storage ==============
# In production, this would be a database (Redis, PostgreSQL, etc.)
# For now, we store in a simple Python dictionary
store = {"session": ChatMessageHistory()}
print("✓ In-memory message history store created")

# ============== STEP 4: Wrap the chain with memory management ==============
# RunnableWithMessageHistory does 3 things:
# 1. Retrieves history before calling LLM
# 2. Inserts history into the prompt
# 3. Saves the new message pair after LLM responds
chain = RunnableWithMessageHistory(
    prompt | llm,
    lambda config: store["session"],      # Function to get history
    input_messages_key="input",           # Name of input field
    history_messages_key="history",       # Name of history field
)
print("✓ Memory-aware chain created")
print()

# ============== STEP 5: Test conversation - Turn 1 ==============
print("--- TURN 1: Introduction ---")
print("User: Hi, I'm Sarvesh!")
response1 = chain.invoke(
    {"input": "Hi, I'm Sarvesh!"},
    config={"configurable": {"session_id": "session"}}
)
print(f"Bot: {response1.content}")
print()

# ============== STEP 6: Test conversation - Turn 2 ==============
print("--- TURN 2: Test Memory (Ask about name) ---")
print("User: What's my name?")
response2 = chain.invoke(
    {"input": "What's my name?"},
    config={"configurable": {"session_id": "session"}}
)
print(f"Bot: {response2.content}")
print()
print("=" * 60)
print("✅ Memory works! The bot remembered the name from Turn 1")
print("=" * 60)


Hi Sarvesh! How can I assist you today?
Your name is Sarvesh! How can I help you today?


---

## 📝 Section 4: Build Your Own Interactive Chatbot

### Your Task

Take the concepts from Sections 1-3 and build an interactive chatbot that:
- ✅ Maintains conversation memory
- ✅ Responds contextually to user input
- ✅ Works in a loop (multi-turn conversation)
- ✅ Can be stopped gracefully

### Code Structure

The template below shows all the pieces you need:
1. LLM initialization
2. Prompt template
3. Message history store
4. Memory-aware chain
5. Input loop

### Important Parameters Explained

- `temperature=0.7`: Controls randomness (0 = deterministic, 1 = very random)
- `input_messages_key="input"`: Tells chain which field has user input
- `history_messages_key="history"`: Tells chain which field gets history
- `session_id`: Identifies which conversation/user

In [ ]:
# =============== SECTION 4: Interactive Chatbot Template ===============

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

print("=" * 60)
print("TEMPLATE: Interactive Chatbot with Memory")
print("=" * 60)
print()

# ========== CONFIGURATION ==========
# You can modify these parameters

# Step 1️⃣: Initialize model with temperature for creativity
# temperature=0.7 means moderate randomness
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
print("✓ Model initialized")

# Step 2️⃣: Create prompt with memory placeholder
# This is the template for every turn
prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("history"),       # Previous messages go here
    ("user", "{input}")                   # Current user message
])
print("✓ Prompt template created")

# Step 3️⃣: Create message history store
# Dictionary structure: {"session_id": ChatMessageHistory()}
# In production: use database
store = {"session": ChatMessageHistory()}
print("✓ History store created")

# Step 4️⃣: Build chain with memory wrapper
# This automatically:
# - Retrieves history
# - Includes it in the prompt
# - Saves new messages
chatbot = RunnableWithMessageHistory(
    prompt | llm,
    lambda config: store["session"],     # Retrieves history for session
    input_messages_key="input",          # Field name for user input
    history_messages_key="history",      # Field name for history
)
print("✓ Memory-aware chatbot chain created")
print()

# ========== INTERACTIVE CHAT LOOP ==========
# Uncomment the code below to run interactive chat
# (Keep commented in notebook to avoid waiting for input)

print("🤖 Chatbot is ready!")
print("Type 'exit' to quit.\n")

# while True:
#     user_text = input("You: ")
#     
#     # Exit on 'exit' or 'quit'
#     if user_text.lower() in ["exit", "quit"]:
#         print("Goodbye! 👋")
#         break
#     
#     # Skip empty inputs
#     if not user_text.strip():
#         continue
#     
#     # Invoke chatbot with user input
#     bot_reply = chatbot.invoke(
#         {"input": user_text},
#         config={"configurable": {"session_id": "session"}}
#     )
#     
#     print("Bot:", bot_reply.content)
#     print()

print("(This cell is commented to avoid blocking notebook execution)")
print("In a real application, uncomment the while loop above to chat interactively")


🤖 Chatbot is ready! Type 'exit' to quit.

Bot: Hi Sarvesh! How can I assist you today?
Bot: Your name is Sarvesh. How can I help you today?
Bot: Got it! You love Python. If you have any questions or topics related to Python that you'd like to discuss, feel free to ask!
Bot: You like Python! Would you like to talk about Python programming or anything specific related to it?
Bot: That's great to know! Chennai is a vibrant city with a rich culture and history. If you have any questions about Chennai, Python, or anything else, feel free to ask!
Bot: Your name is Sarvesh. How can I assist you further?


---

## 📝 Section 5: Testing & Verification Guide

### How to Test Your Chatbot

Use these test cases to verify that your chatbot works correctly and remembers conversation context.

### 🧪 Category 1: Test Memory (Verify context awareness)

**Purpose:** Ensure the chatbot recalls information from earlier in the conversation.

| Turn | User Says | What Chatbot Should Do |
|------|-----------|------------------------|
| 1 | "Hi, my name is Sarvesh." | Acknowledge the introduction |
| 2 | "What's my name?" | Recall "Sarvesh" from Turn 1 |
| 3 | "I love Python." | Acknowledge the preference |
| 4 | "What do I like?" | Recall "Python" from Turn 3 |
| 5 | "I live in Chennai." | Acknowledge the location |
| 6 | "Where do I live?" | Recall "Chennai" from Turn 5 |

**Expected Outcome:** All recall questions answered correctly ✅

---

### 🧠 Category 2: Multi-turn Conversation (Follow-up based on context)

**Purpose:** Show that the chatbot understands context across multiple turns.

| Turn | User Says | What Chatbot Should Do |
|------|-----------|------------------------|
| 1 | "I'm feeling tired today." | Show empathy |
| 2 | "Why do you think I'm tired?" | Reference Turn 1 (you just told me) |
| 3 | "Give me productivity tips." | Provide useful advice given the context |

**Expected Outcome:** Natural conversation flow ✅

---

### 😄 Category 3: Fun & Creative (Engaging demo)

Great for showing the chatbot's personality:

- "Tell me a joke."
- "Give me a fun fact about AI."
- "If you were a superhero, what would be your power?"
- "Write a short poem about learning to code."

**Expected Outcome:** Creative, entertaining responses ✅

---

### 💼 Category 4: Learning-Focused (Educational use)

Demonstrates the chatbot as a learning tool:

- "Explain LangChain in simple terms."
- "Give me 3 project ideas for learning AI."
- "Create a study roadmap for learning Python."
- "What are the best practices in prompt engineering?"
- "Explain the difference between Prompts, Chains, and Memory."

**Expected Outcome:** Informative, well-structured answers ✅

---

### 🗣 Category 5: Complex Conversation (Multi-concept)

Shows advanced conversational ability:

- "What did we talk about earlier?"
- "Do you think AI will replace programming jobs?"
- "Based on what you know about me, what career would suit me?"
- "How can I improve my coding skills given my interests?"

**Expected Outcome:** Context-aware, thoughtful responses ✅

---

### ⚠️ Debugging: What If Memory Doesn't Work?

**Symptoms & Solutions:**

| Problem | Likely Cause | Solution |
|---------|--------------|----------|
| Bot doesn't remember name | History not stored | Check that you're using same `session_id` |
| Bot repeats itself | History includes old responses | This is normal - check if messages are fresh |
| Bot gives generic answers | Missing system prompt | Add system message: `("system", "Be helpful...")` |
| Error about missing API key | OPENAI_API_KEY not set | Check .env file has the key |

---

## 📝 Section 6: Production-Ready Implementation

### What Makes This "Production-Ready"?

A production-ready chatbot should have:
- ✅ **System Prompt**: Clear instructions on how to behave
- ✅ **Proper History Management**: Using functions instead of lambdas
- ✅ **Session Handling**: Support for multiple simultaneous users
- ✅ **Error Handling**: Graceful handling of failures
- ✅ **Configuration Management**: Easy to modify parameters
- ✅ **Logging/Monitoring**: Ability to track conversations

### Code Quality Improvements

1. **Using `get_history()` function instead of lambda**
   - More readable and maintainable
   - Can add error handling inside
   - Follows Python best practices

2. **System Message**
   - Tells LLM how to behave consistently
   - Sets expectations for user interaction
   - Can include domain-specific instructions

3. **Multiple Sessions**
   - Each user gets their own conversation history
   - Session IDs identify different conversations
   - Prevents mixing up context between users

4. **Demonstration Code**
   - Shows actual conversation flow
   - Verifies memory works correctly
   - Easy to test and debug

In [ ]:
# =============== SECTION 6: Production-Ready Chatbot ===============

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

print("=" * 70)
print("🚀 PRODUCTION-READY CHATBOT - Best Practices Demo")
print("=" * 70)
print()

# ============== CONFIGURATION ==============
MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0.7
SESSION_ID = "session1"

print(f"Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Session ID: {SESSION_ID}")
print()

# ============== STEP 1: Initialize LLM ==============
llm = ChatOpenAI(model=MODEL_NAME, temperature=TEMPERATURE)

# ============== STEP 2: Create prompt with system message ==============
# System message tells the LLM how to behave
# This ensures consistent behavior across all turns
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, friendly AI assistant. "
              "Remember all the information the user tells you about themselves "
              "and refer to it in later conversations. "
              "Be concise and clear in your responses."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{input}")
])

# ============== STEP 3: Create message history store ==============
# In production, replace with a real database
# Example: MongoDB, PostgreSQL, Redis, etc.
store = {SESSION_ID: ChatMessageHistory()}

# ============== STEP 4: Define history retrieval function ==============
# Using a function (not lambda) is more maintainable
def get_history(session_id: str):
    """
    Retrieve conversation history for a given session ID.
    
    Args:
        session_id (str): Unique identifier for the conversation
        
    Returns:
        ChatMessageHistory: The conversation history for this session
    """
    if session_id not in store:
        # Auto-create new session if doesn't exist
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# ============== STEP 5: Build memory-aware chatbot ==============
chatbot = RunnableWithMessageHistory(
    prompt | llm,
    get_history,                          # Use our function (not lambda)
    input_messages_key="input",
    history_messages_key="history"
)

# ============== STEP 6: Configuration object ==============
config = {"configurable": {"session_id": SESSION_ID}}

print("✓ Chatbot fully initialized with:")
print("  - System message (defines behavior)")
print("  - Message history storage")
print("  - Memory management")
print()

# ============== DEMONSTRATION ==============
print("=" * 70)
print("📝 Demonstration: 4-Turn Conversation with Memory")
print("=" * 70)
print()

# TURN 1: Introduce name
print("TURN 1: User introduces themselves")
print("-" * 70)
user_input_1 = "Hi, I'm Sarvesh!"
print(f"User: {user_input_1}")
response_1 = chatbot.invoke({"input": user_input_1}, config)
print(f"Bot: {response_1.content}")
print()

# TURN 2: Share a preference
print("TURN 2: User shares a preference")
print("-" * 70)
user_input_2 = "My favorite language is Python."
print(f"User: {user_input_2}")
response_2 = chatbot.invoke({"input": user_input_2}, config)
print(f"Bot: {response_2.content}")
print()

# TURN 3: Test memory - recall favorite language
print("TURN 3: Test Memory - Ask about favorite language")
print("-" * 70)
user_input_3 = "What's my favorite language?"
print(f"User: {user_input_3}")
response_3 = chatbot.invoke({"input": user_input_3}, config)
print(f"Bot: {response_3.content}")
print(f"✓ Memory works! Bot recalled 'Python' from Turn 2")
print()

# TURN 4: Test memory - recall name
print("TURN 4: Test Memory - Ask about name")
print("-" * 70)
user_input_4 = "What is my name?"
print(f"User: {user_input_4}")
response_4 = chatbot.invoke({"input": user_input_4}, config)
print(f"Bot: {response_4.content}")
print(f"✓ Memory works! Bot recalled 'Sarvesh' from Turn 1")
print()

# ============== VERIFICATION ==============
print("=" * 70)
print("✅ VERIFICATION COMPLETE")
print("=" * 70)
print()
print("The chatbot successfully:")
print("  ✓ Responded to user introduction")
print("  ✓ Acknowledged preferences")
print("  ✓ Recalled favorite language (Python)")
print("  ✓ Recalled user name (Sarvesh)")
print()
print("🎓 Key Learning:")
print("  - Memory is maintained across turns")
print("  - System message provides consistent behavior")
print("  - Each session has isolated conversation history")
print("  - Production code is clean and maintainable")
print()



User: Hi, I'm Sarvesh!
Bot: content='Hi Sarvesh! How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 25, 'total_tokens': 37, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-Cc8xvgcqgzky7FGhTF2GMN3lv5HeR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--7ba51dea-f218-482b-ad29-bf64f7f8cd95-0' usage_metadata={'input_tokens': 25, 'output_tokens': 12, 'total_tokens': 37, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

User: My favorite language is Python.
Bot: content="That's great to hear! Python is a versatile and powerful langua

---

## 🎓 Summary & Key Takeaways

### What You Learned

| Concept | What It Does | Why It Matters |
|---------|-------------|----------------|
| **Prompts** | Structured templates for communicating with LLMs | Reusable, maintainable, professional |
| **Chains** | Connect multiple operations (prompts, LLMs, functions) | Build complex workflows not just Q&A |
| **Memory** | Track conversation history and context | Create chatbots that feel natural & aware |

### The Learning Journey

1. **Section 1**: Started simple → Basic prompt template
2. **Section 2**: Added complexity → Chained two prompts together
3. **Section 3**: Added state → Introduced memory to chains
4. **Section 4**: Made it interactive → Full chatbot loop
5. **Section 5**: Tested thoroughly → Verification guide
6. **Section 6**: Went production → Best practices code

### Key Code Patterns to Remember

**Pattern 1: Simple Prompt**
```python
prompt = PromptTemplate.from_template("Summarize: {content}")
output = prompt.format(content="...")
```

**Pattern 2: Chain Operations**
```python
chain = prompt | llm | another_prompt | llm
result = chain.invoke({"input": "..."})
```

**Pattern 3: Add Memory to Chain**
```python
chatbot = RunnableWithMessageHistory(
    prompt | llm,
    get_history,
    input_messages_key="input",
    history_messages_key="history"
)
```

### Common Mistakes to Avoid

❌ **Mistake 1**: Forgetting to load your API key
✅ **Solution**: Always call `load_dotenv()` first

❌ **Mistake 2**: Using wrong session ID across turns
✅ **Solution**: Keep session ID consistent in config

❌ **Mistake 3**: Including system message only in first turn
✅ **Solution**: Include it in `ChatPromptTemplate` (appears every turn)

❌ **Mistake 4**: Not testing memory properly
✅ **Solution**: Use the test categories from Section 5

### Next Steps

🚀 **Try building:**
1. A customer support chatbot with memory
2. A code explanation chatbot
3. A brainstorming partner that remembers your ideas
4. A language learning chatbot

📚 **Explore further:**
- Add database support (MongoDB, PostgreSQL)
- Add conversation search/retrieval
- Add streaming responses
- Add multi-user support
- Add conversation cleanup/expiration

---

## 📝 Recap: 3 Core Concepts

### 1. Prompts = Structured Instructions
How you ask the LLM to do something. Better prompts = better results.

### 2. Chains = Workflow Composition
Connect prompts, LLMs, and functions together for complex tasks.

### 3. Memory = Conversational Context
Track what was said so the chatbot understands the big picture.

**Together**: Prompts + Chains + Memory = Conversational AI Applications 🎉